# Simulasi & Eksperimen Hyperparameter Tuning (Dataset Asli)

Notebook ini dibuat khusus untuk memisahkan **proses pencarian parameter terbaik (Hyperparameter Tuning)** dari notebook utama. Notebook ini menggunakan **dataset fitur batik asli** melalui tahapan ekstraksi GLCM dan CNN (MobileNetV2) yang sama persis dengan notebook utama.


In [ ]:
# Import library yang dibutuhkan
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

# Library Ekstraksi Fitur Tekstur
from skimage.feature import graycomatrix, graycoprops

# Library Deep Learning & Preprocessing
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

# Library Klasifikasi, Evaluasi, & Tuning
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV


## 1. Persiapan Data (Ekstraksi Fitur Hybrid & PCA)

Proses ini identik dengan notebook utama: memuat gambar, pemerataan pencahayaan CLAHE, augmentasi rotasi, zero-padding, ekstraksi fitur GLCM & CNN, normalisasi, dan reduksi dimensi (PCA). Karena kita memproses data asli, tahap ini akan memakan waktu.


In [ ]:
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'
IMG_SIZE = (224, 224) 

# Load CNN
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
base_model.trainable = False 

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

def apply_zero_padding(img, target_size=(224, 224)):
    old_size = img.shape[:2]
    ratio = min(target_size[0]/old_size[0], target_size[1]/old_size[1])
    new_size = tuple([int(x*ratio) for x in old_size])
    img_resized = cv2.resize(img, (new_size[1], new_size[0]))
    delta_w = target_size[1] - new_size[1]
    delta_h = target_size[0] - new_size[0]
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)
    color = [0, 0, 0]
    return cv2.copyMakeBorder(img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)

def extract_glcm_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    distances = [1]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(gray, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    ASM = graycoprops(glcm, 'ASM').mean()
    return [contrast, dissimilarity, homogeneity, energy, correlation, ASM]

def extract_cnn_features(img, model):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_padded = apply_zero_padding(img_rgb, target_size=IMG_SIZE)
    img_array = img_to_array(img_padded)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    features = model.predict(img_array, verbose=0)
    return features.flatten()

def process_dataset(directory, cnn_model, augment_rotation=False):
    features_combined = []
    labels = []
    classes = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])
    
    for class_name in tqdm(classes, desc=f"Processing {directory}"):
        class_dir = os.path.join(directory, class_name)
        for img_name in os.listdir(class_dir):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue
                
            img_clahe = apply_clahe(img)
            rotations = [None]
            if augment_rotation:
                rotations.extend([cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_180, cv2.ROTATE_90_COUNTERCLOCKWISE])
                
            for rot in rotations:
                try:
                    img_processed = cv2.rotate(img_clahe, rot) if rot is not None else img_clahe
                    glcm_feats = extract_glcm_features(img_processed)
                    cnn_feats = extract_cnn_features(img_processed, cnn_model)
                    combined = np.hstack((glcm_feats, cnn_feats))
                    features_combined.append(combined)
                    labels.append(class_name)
                except Exception as e:
                    pass
    return np.array(features_combined), np.array(labels)

print("--- Ekstraksi Fitur Latih (Augmentasi) ---")
X_train, y_train_raw = process_dataset(TRAIN_DIR, base_model, augment_rotation=True)
print("\n--- Ekstraksi Fitur Uji ---")
X_test, y_test_raw = process_dataset(TEST_DIR, base_model, augment_rotation=False)

# Preprocessing
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"\nProses PCA selesai. Dimensi fitur setelah PCA: {X_train_pca.shape[1]}")


## 2. Tuning Otomatis Menggunakan GridSearchCV

Mencari parameter terbaik untuk berbagai model menggunakan teknik *Cross-Validation*.


In [9]:
model_params = {
    "SVM": {
        "model": SVC(random_state=42),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf"]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7],
            "weights": ["uniform", "distance"]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100],
            "max_depth": [None, 10, 20]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [None, 10, 20, 30]
        }
    },
    "Dense Neural Network (MLP)": {
        "model": MLPClassifier(random_state=42),
        "params": {
            "hidden_layer_sizes": [(128,), (256, 128)],
            "max_iter": [500]
        }
    }
}

grid_results = []

for name, mp in model_params.items():
    print(f"Tuning Otomatis model {name}...")
    clf = GridSearchCV(mp["model"], mp["params"], cv=3, n_jobs=-1, verbose=0)
    
    start_train = time.time()
    clf.fit(X_train_pca, y_train)
    end_train = time.time()
    
    y_pred = clf.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    
    grid_results.append({
        'Model': name,
        'Parameter Terbaik (GridSearchCV)': str(clf.best_params_),
        'Akurasi Uji (Test)': f"{acc*100:.2f}%",
        'Waktu Eksekusi': f"{end_train - start_train:.2f} detik"
    })

print("\nHASIL TUNING OTOMATIS PADA FITUR BATIK:")
display(pd.DataFrame(grid_results))


Tuning Otomatis model SVM...
Tuning Otomatis model KNN...
Tuning Otomatis model Random Forest...
Tuning Otomatis model Decision Tree...
Tuning Otomatis model Dense Neural Network (MLP)...

HASIL TUNING OTOMATIS PADA FITUR BATIK:


,Model,Parameter Terbaik (GridSearchCV),Akurasi Uji (Test),Waktu Eksekusi
0,SVM,"{'C': 0.1, 'kernel': 'linear'}",95.94%,22.58 detik
1,KNN,"{'n_neighbors': 3, 'weights': 'distance'}",93.75%,3.85 detik
2,Random Forest,"{'max_depth': 20, 'n_estimators': 100}",91.25%,33.60 detik
3,Decision Tree,{'max_depth': None},56.25%,11.67 detik
4,Dense Neural Network (MLP),"{'hidden_layer_sizes': (256, 128), 'max_iter':...",95.62%,21.58 detik


## 3. Tuning Manual (Membongkar Logika di Balik GridSearchCV)

Sebagai demonstrasi, berikut adalah logika pencarian kombinasi parameter menggunakan struktur *Nested Loop* (Perulangan berjenjang) secara manual pada model SVM dan KNN. Hasil dari tuning manual ini akan sama persis dengan metode otomatis karena dieksekusi pada kumpulan fitur yang sama.


In [10]:
print("--- TUNING MANUAL SVM ---")

C_values = [0.1, 1, 10]
kernels = ['linear', 'rbf']

best_svm_acc = 0
best_svm_params = {}
svm_results = []

for c in C_values:
    for k in kernels:
        model = SVC(C=c, kernel=k, random_state=42)
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)
        acc = accuracy_score(y_test, y_pred)
        
        svm_results.append({'C': c, 'kernel': k, 'Akurasi': acc})
        
        if acc > best_svm_acc:
            best_svm_acc = acc
            best_svm_params = {'C': c, 'kernel': k}

print(f"\nParameter SVM Terbaik Manual: {best_svm_params} dengan Akurasi {best_svm_acc*100:.2f}%")
display(pd.DataFrame(svm_results))


print("\n--- TUNING MANUAL KNN ---")
neighbors_values = [3, 5, 7]
weights_options = ['uniform', 'distance']

best_knn_acc = 0
best_knn_params = {}
knn_results = []

for n in neighbors_values:
    for w in weights_options:
        model = KNeighborsClassifier(n_neighbors=n, weights=w)
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)
        acc = accuracy_score(y_test, y_pred)

        knn_results.append({'n_neighbors': n, 'weights': w, 'Akurasi': acc})

        if acc > best_knn_acc:
            best_knn_acc = acc
            best_knn_params = {'n_neighbors': n, 'weights': w}

print(f"\nParameter KNN Terbaik Manual: {best_knn_params} dengan Akurasi {best_knn_acc*100:.2f}%")
display(pd.DataFrame(knn_results))


print("\n--- TUNING MANUAL RANDOM FOREST ---")
n_estimators_values = [50, 100]
max_depth_rf_values = [None, 10, 20]

best_rf_acc = 0
best_rf_params = {}
rf_results = []

for n in n_estimators_values:
    for d in max_depth_rf_values:
        model = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)
        acc = accuracy_score(y_test, y_pred)

        rf_results.append({'n_estimators': n, 'max_depth': d, 'Akurasi': acc})

        if acc > best_rf_acc:
            best_rf_acc = acc
            best_rf_params = {'n_estimators': n, 'max_depth': d}

print(f"\nParameter Random Forest Terbaik Manual: {best_rf_params} dengan Akurasi {best_rf_acc*100:.2f}%")
display(pd.DataFrame(rf_results))


print("\n--- TUNING MANUAL DECISION TREE ---")
max_depth_dt_values = [None, 10, 20, 30]

best_dt_acc = 0
best_dt_params = {}
dt_results = []

for d in max_depth_dt_values:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)

    dt_results.append({'max_depth': d, 'Akurasi': acc})

    if acc > best_dt_acc:
        best_dt_acc = acc
        best_dt_params = {'max_depth': d}

print(f"\nParameter Decision Tree Terbaik Manual: {best_dt_params} dengan Akurasi {best_dt_acc*100:.2f}%")
display(pd.DataFrame(dt_results))


print("\n--- TUNING MANUAL DENSE NEURAL NETWORK (MLP) ---")
hidden_layers_values = [(128,), (256, 128)]
max_iter_values = [500]

best_mlp_acc = 0
best_mlp_params = {}
mlp_results = []

for hl in hidden_layers_values:
    for mi in max_iter_values:
        model = MLPClassifier(hidden_layer_sizes=hl, max_iter=mi, random_state=42)
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)
        acc = accuracy_score(y_test, y_pred)

        mlp_results.append({'hidden_layer_sizes': hl, 'max_iter': mi, 'Akurasi': acc})

        if acc > best_mlp_acc:
            best_mlp_acc = acc
            best_mlp_params = {'hidden_layer_sizes': hl, 'max_iter': mi}

print(f"\nParameter Dense Neural Network (MLP) Terbaik Manual: {best_mlp_params} dengan Akurasi {best_mlp_acc*100:.2f}%")
display(pd.DataFrame(mlp_results))


--- TUNING MANUAL SVM ---

Parameter SVM Terbaik Manual: {'C': 0.1, 'kernel': 'linear'} dengan Akurasi 95.94%


,C,kernel,Akurasi
0,0.1,linear,0.959375
1,0.1,rbf,0.775000
2,1.0,linear,0.959375
3,1.0,rbf,0.959375
4,10.0,linear,0.959375
5,10.0,rbf,0.959375



--- TUNING MANUAL KNN ---

Parameter KNN Terbaik Manual: {'n_neighbors': 3, 'weights': 'distance'} dengan Akurasi 93.75%


,n_neighbors,weights,Akurasi
0,3,uniform,0.931250
1,3,distance,0.937500
2,5,uniform,0.915625
3,5,distance,0.931250
4,7,uniform,0.900000
5,7,distance,0.931250



--- TUNING MANUAL RANDOM FOREST ---

Parameter Random Forest Terbaik Manual: {'n_estimators': 100, 'max_depth': None} dengan Akurasi 92.19%


,n_estimators,max_depth,Akurasi
0,50,NaN,0.878125
1,50,10.0,0.837500
2,50,20.0,0.893750
3,100,NaN,0.921875
4,100,10.0,0.881250
5,100,20.0,0.912500



--- TUNING MANUAL DECISION TREE ---

Parameter Decision Tree Terbaik Manual: {'max_depth': 20} dengan Akurasi 59.06%


,max_depth,Akurasi
0,NaN,0.562500
1,10.0,0.521875
2,20.0,0.590625
3,30.0,0.562500



--- TUNING MANUAL DENSE NEURAL NETWORK (MLP) ---

Parameter Dense Neural Network (MLP) Terbaik Manual: {'hidden_layer_sizes': (128,), 'max_iter': 500} dengan Akurasi 96.56%


,hidden_layer_sizes,max_iter,Akurasi
0,"(128,)",500,0.965625
1,"(256, 128)",500,0.956250
